In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
propanolamine,75.11,3.86626,3.006102302,180.4056401,2,2
"""
assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
propanolamine,H,propanolamine,e,2000.0,0.03
"""


model = PCSAFT(["propanolamine"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_propanolamine.csv")
fix_line_endings("rhol_propanolamine.csv")

Fixed: satp_propanolamine.csv
Fixed: rhol_propanolamine.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

println(my_liquid_density(model, 293.15))
println(my_saturation_p(model, 273.15))

976.1844586341307
56.830969911280064


In [5]:
toestimate = [
    Dict(
        :param => :segment,
        :lower => 1.0,
        :upper => 10.0,
        :guess => 3.
    ),
    Dict(
        :param => :epsilon,
        :lower => 100.,
        :upper => 500.,
        :guess => 250.
    ),
    Dict(
        :param => :sigma,
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 7.0,
        :guess => 5.
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

5-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 10.0, :param => :segment, :guess => 3.0, :lower => 1.0)
 Dict(:upper => 500.0, :param => :epsilon, :guess => 250.0, :lower => 100.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :upper => 7.0, :guess => 5.0, :lower => 2.0)
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_propanolamine.csv",
        "satp_propanolamine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 1.4860779469932797


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([4.819752026944481, 185.60362971972057, 2.789719497655879, 1895.586390816829, 0.1637576437125451], PCSAFT{BasicIdeal, Float64}("propanolamine"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_propanolamine.csv",   my_saturation_p)

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



=== AAD: satp_propanolamine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
360.8600    1333.508556   1338.379297   0.3653  
363.3694    1544.627360   1548.464579   0.2484  
365.8788    1784.469467   1787.136651   0.1495  
368.3882    2056.280729   2057.652576   0.0667  
370.8976    2363.594172   2363.561382   0.0014  
373.4069    2710.247477   2708.721882   0.0563  
375.9163    3100.400900   3097.320853   0.0993  
378.4257    3538.555602   3533.891547   0.1318  
380.9351    4029.572360   4023.332471   0.1549  
383.4445    4578.690622   4570.926405   0.1696  
385.9539    5191.547890   5182.359587   0.1770  
388.4633    5874.199371   5863.741033   0.1780  
390.9727    6633.137890   6621.621932   0.1736  
393.4820    7475.314007   7463.015066   0.1645  
395.9914    8408.156303   8395.414190   0.1515  
398.5008    9439.591816   9426.813343   0.1354  
401.0102    10578.066560  10565.726015  0.1167  
403.5196    11832.566116  11821.204133  0.0960  
406.0290    13212.63623

0.11866083166013683

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_propanolamine.csv",   my_liquid_density)


=== AAD: rhol_propanolamine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
293.1500    987.650000    987.852281    0.0205  
303.1500    979.650000    979.721018    0.0072  
313.1500    971.610000    971.590234    0.0020  
323.1500    963.520000    963.432850    0.0090  
333.1500    955.380000    955.223787    0.0164  
AARD = 0.0110%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.011032131174658231